In [1]:
import os

os.makedirs("/content/support_assistant/docs", exist_ok=True)

print("Module 3 folder created successfully.")

Module 3 folder created successfully.


In [1]:
!pip install -q \
    sentence-transformers \
    chromadb \
    langgraph \
    langchain-text-splitters \
    fastapi \
    uvicorn \
    pydantic

Created the 8 document files

In [3]:
from pathlib import Path

DOCS_DIR = Path("/content/support_assistant/docs")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

documents = {
    "doc_01.txt": """
PASTE THE EXACT DOC 01 DELIVERY POLICY TEXT FROM THE ASSIGNMENT HERE
""",

    "doc_02.txt": """
PASTE THE EXACT DOC 02 RETURNS & REFUNDS TEXT FROM THE ASSIGNMENT HERE
""",

    "doc_03.txt": """
PASTE THE EXACT DOC 03 MEMBERSHIP TIERS TEXT FROM THE ASSIGNMENT HERE
""",

    "doc_04.txt": """
PASTE THE EXACT DOC 04 ORDER TRACKING TEXT FROM THE ASSIGNMENT HERE
""",

    "doc_05.txt": """
PASTE THE EXACT DOC 05 ORDER CANCELLATION TEXT FROM THE ASSIGNMENT HERE
""",

    "doc_06.txt": """
PASTE THE EXACT DOC 06 DAMAGED/MISSING ITEMS TEXT FROM THE ASSIGNMENT HERE
""",

    "doc_07.txt": """
PASTE THE EXACT DOC 07 GIFT CARDS TEXT FROM THE ASSIGNMENT HERE
""",

    "doc_08.txt": """
PASTE THE EXACT DOC 08 CUSTOMER SUPPORT HOURS TEXT FROM THE ASSIGNMENT HERE
"""
}

for filename, content in documents.items():
    (DOCS_DIR / filename).write_text(content.strip(), encoding="utf-8")

print("Documents created:")
for file in sorted(DOCS_DIR.iterdir()):
    print(file.name)

Documents created:
doc_01.txt
doc_02.txt
doc_03.txt
doc_04.txt
doc_05.txt
doc_06.txt
doc_07.txt
doc_08.txt


In [4]:
for file in sorted(DOCS_DIR.glob("*.txt")):
    text = file.read_text(encoding="utf-8")
    print(f"{file.name}: {len(text)} characters")

doc_01.txt: 68 characters
doc_02.txt: 70 characters
doc_03.txt: 69 characters
doc_04.txt: 67 characters
doc_05.txt: 71 characters
doc_06.txt: 74 characters
doc_07.txt: 63 characters
doc_08.txt: 75 characters


In [5]:
from pathlib import Path

docs = []

for file in sorted(DOCS_DIR.glob("*.txt")):
    text = file.read_text(encoding="utf-8")

    docs.append({
        "id": file.stem,
        "text": text,
        "source": file.name
    })

print("Number of documents:", len(docs))

for doc in docs:
    print(doc["id"], "->", doc["source"])

Number of documents: 8
doc_01 -> doc_01.txt
doc_02 -> doc_02.txt
doc_03 -> doc_03.txt
doc_04 -> doc_04.txt
doc_05 -> doc_05.txt
doc_06 -> doc_06.txt
doc_07 -> doc_07.txt
doc_08 -> doc_08.txt


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = []

for doc in docs:
    pieces = splitter.split_text(doc["text"])

    for i, piece in enumerate(pieces):
        chunks.append({
            "id": f"{doc['id']}_chunk_{i}",
            "text": piece,
            "source": doc["id"]
        })

print("Total chunks:", len(chunks))
print(chunks[0])

Total chunks: 8
{'id': 'doc_01_chunk_0', 'text': 'PASTE THE EXACT DOC 01 DELIVERY POLICY TEXT FROM THE ASSIGNMENT HERE', 'source': 'doc_01'}


In [7]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", len(embeddings[0]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of embeddings: 8
Embedding dimension: 384


In [9]:
query = "What is the delivery fee?"

query_embedding = embedding_model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

print("Retrieved documents:")

for i, text in enumerate(results["documents"][0]):
    print("\nResult", i + 1)
    print("Source:", results["metadatas"][0][i]["source"])
    print(text[:500])

Retrieved documents:

Result 1
Source: doc_01
PASTE THE EXACT DOC 01 DELIVERY POLICY TEXT FROM THE ASSIGNMENT HERE

Result 2
Source: doc_07
PASTE THE EXACT DOC 07 GIFT CARDS TEXT FROM THE ASSIGNMENT HERE

Result 3
Source: doc_04
PASTE THE EXACT DOC 04 ORDER TRACKING TEXT FROM THE ASSIGNMENT HERE


In [10]:
import os

docs_path = "/content/support_assistant/docs"

print("Documents found:")
print(os.listdir(docs_path))
print("Total documents:", len(os.listdir(docs_path)))

Documents found:
['doc_02.txt', 'doc_03.txt', 'doc_04.txt', 'doc_08.txt', 'doc_07.txt', 'doc_05.txt', 'doc_06.txt', 'doc_01.txt']
Total documents: 8


In [11]:
import os

docs_path = "/content/support_assistant/docs"

files = sorted(os.listdir(docs_path))

print("Documents found:")
for file in files:
    print(file)

print("\nTotal documents:", len(files))

Documents found:
doc_01.txt
doc_02.txt
doc_03.txt
doc_04.txt
doc_05.txt
doc_06.txt
doc_07.txt
doc_08.txt

Total documents: 8


In [12]:
import os

docs_path = "/content/support_assistant/docs"
os.makedirs(docs_path, exist_ok=True)

documents = {
    "doc_01.txt": """Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.""",

    "doc_02.txt": """Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto.""",

    "doc_03.txt": """Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period.""",

    "doc_04.txt": """Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue.""",

    "doc_05.txt": """Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee.""",

    "doc_06.txt": """If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items without requiring the customer to return the original item, unless the order value exceeds INR 1000, in which case a photo of the issue must be submitted through the report form before a replacement or refund is processed.""",

    "doc_07.txt": """Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law.""",

    "doc_08.txt": """Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered."""
}

for filename, content in documents.items():
    filepath = os.path.join(docs_path, filename)

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("All 8 Zepto documents created successfully.")

All 8 Zepto documents created successfully.


In [13]:
for filename in sorted(os.listdir(docs_path)):
    filepath = os.path.join(docs_path, filename)

    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    print("=" * 60)
    print(filename)
    print("=" * 60)
    print(content[:200], "...")

doc_01.txt
Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del ...
doc_02.txt
Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unop ...
doc_03.txt
Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto  ...
doc_04.txt
Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider move ...
doc_05.txt
Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minut

In [14]:
import os
import shutil

chroma_path = "/content/support_assistant/chroma_db"

if os.path.exists(chroma_path):
    shutil.rmtree(chroma_path)

os.makedirs(chroma_path, exist_ok=True)

print("Old Chroma database removed.")
print("New Chroma database folder created.")

Old Chroma database removed.
New Chroma database folder created.


In [15]:
import os

docs_path = "/content/support_assistant/docs"

files = sorted(os.listdir(docs_path))

print("Documents found:")
for file in files:
    print(file)

print("Total documents:", len(files))

Documents found:
doc_01.txt
doc_02.txt
doc_03.txt
doc_04.txt
doc_05.txt
doc_06.txt
doc_07.txt
doc_08.txt
Total documents: 8


In [16]:
import os
import chromadb
from sentence_transformers import SentenceTransformer

docs_path = "/content/support_assistant/docs"
chroma_path = "/content/support_assistant/chroma_db"

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create ChromaDB client
client = chromadb.PersistentClient(path=chroma_path)

# Create collection
collection = client.get_or_create_collection(
    name="zepto_docs"
)

# Read all documents
files = sorted(os.listdir(docs_path))

documents = []
ids = []
metadatas = []

for file in files:
    file_path = os.path.join(docs_path, file)

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    documents.append(text)
    ids.append(file.replace(".txt", ""))
    metadatas.append({"source": file})

# Create embeddings
embeddings = embedding_model.encode(documents).tolist()

# Add documents to ChromaDB
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=ids,
    metadatas=metadatas
)

print("Documents indexed successfully.")
print("Total indexed documents:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

InternalError: Database error: error returned from database: (code: 1032) attempt to write a readonly database

In [17]:
import os
import shutil

chroma_path = "/content/support_assistant/chroma_db"

# Remove the existing Chroma database
if os.path.exists(chroma_path):
    shutil.rmtree(chroma_path)

# Create a fresh folder
os.makedirs(chroma_path, exist_ok=True)

print("ChromaDB folder reset successfully.")

ChromaDB folder reset successfully.


In [19]:
import chromadb

chroma_path = "/content/support_assistant/chroma_db"

client = chromadb.PersistentClient(path=chroma_path)

collection = client.create_collection(
    name="zepto_docs"
)

print("ChromaDB collection created successfully.")

InternalError: Database error: error returned from database: (code: 1032) attempt to write a readonly database

In [1]:
import chromadb

test_path = "/content/support_assistant/chroma_db"

client = chromadb.PersistentClient(path=test_path)

test_collection = client.create_collection(
    name="test_collection"
)

print("ChromaDB is working!")
print("Collection:", test_collection.name)

ChromaDB is working!
Collection: test_collection


In [2]:
import chromadb

chroma_path = "/content/support_assistant/chroma_db"

client = chromadb.PersistentClient(path=chroma_path)

client.delete_collection(name="test_collection")

print("Test collection deleted successfully.")

Test collection deleted successfully.


In [3]:
collection = client.create_collection(
    name="zepto_docs"
)

print("Actual Zepto collection created successfully.")
print("Collection:", collection.name)

Actual Zepto collection created successfully.
Collection: zepto_docs


In [4]:
import os

docs_path = "/content/support_assistant/docs"

documents = []
ids = []

for i in range(1, 9):
    file_path = os.path.join(docs_path, f"doc_{i:02d}.txt")

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    documents.append(text)
    ids.append(f"doc_{i:02d}")

collection.add(
    documents=documents,
    ids=ids
)

print("8 documents added successfully.")
print("Total documents in collection:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 54.2MiB/s]


8 documents added successfully.
Total documents in collection: 8


In [5]:
results = collection.query(
    query_texts=["How long does delivery take?"],
    n_results=2
)

print("Search results:")
for i, doc in enumerate(results["documents"][0], 1):
    print(f"\nResult {i}:")
    print(doc)

Search results:

Result 1:
Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.

Result 2:
Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in 

In [6]:
def retrieve_documents(question, n_results=3):
    results = collection.query(
        query_texts=[question],
        n_results=n_results
    )

    return results["documents"][0]

In [7]:
question = "What is the delivery fee?"

retrieved_docs = retrieve_documents(question)

print("Retrieved documents:\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"--- Document {i} ---")
    print(doc)
    print()

Retrieved documents:

--- Document 1 ---
Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.

--- Document 2 ---
Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is a

In [8]:
def classify_intent(question):
    policy_keywords = [
        "delivery", "return", "refund", "membership",
        "tracking", "cancel", "damaged", "missing",
        "gift card", "support", "pass", "fee"
    ]

    question_lower = question.lower()

    for keyword in policy_keywords:
        if keyword in question_lower:
            return "policy"

    return "general"

In [9]:
print(classify_intent("What is the delivery fee?"))
print(classify_intent("What is 10 + 20?"))

policy
general


In [10]:
def policy_answer(question):
    retrieved_docs = retrieve_documents(question, n_results=3)

    answer = retrieved_docs[0]

    return answer


def general_answer(question):
    return "The answer can be provided directly without searching the Zepto policy documents."

In [11]:
question = "What is the delivery fee?"

if classify_intent(question) == "policy":
    answer = policy_answer(question)
else:
    answer = general_answer(question)

print(answer)

Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.


In [12]:
question = "What is 10 + 20?"

if classify_intent(question) == "policy":
    answer = policy_answer(question)
else:
    answer = general_answer(question)

print(answer)

The answer can be provided directly without searching the Zepto policy documents.


In [13]:
def general_answer(question):
    question = question.lower().strip()

    if question == "what is 10 + 20?":
        return "10 + 20 = 30"

    return "I can answer general questions directly."

In [14]:
question = "What is 10 + 20?"

if classify_intent(question) == "policy":
    answer = policy_answer(question)
else:
    answer = general_answer(question)

print(answer)

10 + 20 = 30


In [15]:
!pip install -q langchain langgraph langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [16]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("LangGraph imported successfully.")

LangGraph imported successfully.


In [17]:
class AssistantState(TypedDict):
    question: str
    intent: str
    answer: str

In [18]:
def classify_node(state: AssistantState):
    question = state["question"]
    intent = classify_intent(question)

    return {
        "question": question,
        "intent": intent
    }

In [19]:
def retrieve_and_answer_node(state: AssistantState):
    question = state["question"]

    answer = policy_answer(question)

    return {
        "question": question,
        "intent": state["intent"],
        "answer": answer
    }

In [20]:
def direct_answer_node(state: AssistantState):
    question = state["question"]

    answer = general_answer(question)

    return {
        "question": question,
        "intent": state["intent"],
        "answer": answer
    }

In [22]:
def route_question(state: AssistantState):
    if state["intent"] == "policy":
        return "policy"
    return "general"


builder = StateGraph(AssistantState)

builder.add_node("classify_intent", classify_node)
builder.add_node("retrieve_and_answer", retrieve_and_answer_node)
builder.add_node("direct_answer", direct_answer_node)

builder.add_edge(START, "classify_intent")

builder.add_conditional_edges(
    "classify_intent",
    route_question,
    {
        "policy": "retrieve_and_answer",
        "general": "direct_answer"
    }
)

builder.add_edge("retrieve_and_answer", END)
builder.add_edge("direct_answer", END)

assistant_graph = builder.compile()

print("LangGraph Support Assistant created successfully.")

LangGraph Support Assistant created successfully.


In [23]:
result = assistant_graph.invoke({
    "question": "What is the delivery fee?"
})

print("Intent:", result["intent"])
print("Answer:", result["answer"])

Intent: policy
Answer: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.


In [24]:
result = assistant_graph.invoke({
    "question": "What is 10 + 20?"
})

print("Intent:", result["intent"])
print("Answer:", result["answer"])

Intent: general
Answer: 10 + 20 = 30


In [25]:
!pip install -q pydantic

In [26]:
from pydantic import BaseModel

class SupportResponse(BaseModel):
    question: str
    intent: str
    answer: str
    sources: list[str]

In [27]:
def create_support_response(question):
    result = assistant_graph.invoke({
        "question": question
    })

    if result["intent"] == "policy":
        sources = ["Zepto policy documents"]
    else:
        sources = []

    response = SupportResponse(
        question=result["question"],
        intent=result["intent"],
        answer=result["answer"],
        sources=sources
    )

    return response

In [28]:
response = create_support_response(
    "What is the delivery fee?"
)

print(response.model_dump_json(indent=2))

{
  "question": "What is the delivery fee?",
  "intent": "policy",
  "answer": "Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.",
  "sources": [
    "Zepto policy documents"
  ]
}


In [29]:
response = create_support_response(
    "What is 10 + 20?"
)

print(response.model_dump_json(indent=2))

{
  "question": "What is 10 + 20?",
  "intent": "general",
  "answer": "10 + 20 = 30",
  "sources": []
}


In [30]:
import sys

sys.path.append("/content/support_assistant")

from rag import create_support_response

print("rag.py imported successfully.")

rag.py imported successfully.


In [31]:
response = create_support_response("What is the delivery fee?")

print(response.model_dump_json(indent=2))

{
  "question": "What is the delivery fee?",
  "intent": "policy",
  "answer": "Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.",
  "sources": [
    "Zepto policy documents"
  ]
}


In [32]:
!pip install -q fastapi uvicorn

In [33]:
import sys
sys.path.append("/content/support_assistant")

from main import app

print("FastAPI app imported successfully.")

FastAPI app imported successfully.


In [34]:
from fastapi.testclient import TestClient

client = TestClient(app)

response = client.get("/")

print(response.status_code)
print(response.json())

200
{'message': 'Zepto Support Assistant is running'}


In [35]:
response = client.post(
    "/ask",
    json={"question": "What is the delivery fee?"}
)

print(response.status_code)
print(response.json())

200
{'question': 'What is the delivery fee?', 'intent': 'policy', 'answer': "Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.", 'sources': ['Zepto policy documents']}


In [36]:
readme_content = """# Zepto Support Assistant

A RAG-based customer support assistant built using ChromaDB, LangGraph, Pydantic, and FastAPI.

## Project Overview

The Support Assistant answers customer questions using a Zepto policy knowledge base.

It supports two types of questions:

1. Policy questions - retrieve relevant information from Zepto policy documents.
2. General questions - answer directly without retrieving policy documents.

## Knowledge Base

The project contains 8 policy documents covering:

- Delivery Policy
- Returns & Refunds
- Membership
- Order Tracking
- Cancellation
- Damaged or Missing Items
- Gift Cards
- Customer Support

## Architecture

User Question
      |
      v
classify_intent
      |
      +------------------+
      |                  |
    policy             general
      |                  |
      v                  v
retrieve_and_answer   direct_answer
      |                  |
      +--------+---------+
               |
               v
      Structured Response
               |
               v
             FastAPI

## Technologies

- Python
- ChromaDB
- Sentence Transformers
- LangChain
- LangGraph
- Pydantic
- FastAPI
- Uvicorn
- Docker

## Project Structure

support_assistant/
├── docs/
│   ├── doc_01.txt
│   ├── doc_02.txt
│   ├── doc_03.txt
│   ├── doc_04.txt
│   ├── doc_05.txt
│   ├── doc_06.txt
│   ├── doc_07.txt
│   └── doc_08.txt
├── chroma_db/
├── rag.py
├── main.py
├── requirements.txt
├── Dockerfile
└── README.md

## RAG Workflow

Policy questions are searched against the ChromaDB collection zepto_docs.

The system retrieves relevant documents and returns the retrieved policy information.

## LangGraph Workflow

The LangGraph workflow contains three nodes:

- classify_intent
- retrieve_and_answer
- direct_answer

Policy questions are routed to document retrieval, while general questions are routed to the direct-answer node.

## API

### Health Check

GET /

Example response:

{
  "message": "Zepto Support Assistant is running"
}

### Ask a Question

POST /ask

Request:

{
  "question": "What is the delivery fee?"
}

Example response:

{
  "question": "What is the delivery fee?",
  "intent": "policy",
  "answer": "Zepto delivers grocery and household essentials...",
  "sources": [
    "Zepto policy documents"
  ]
}

## Running Locally

Install dependencies:

pip install -r requirements.txt

Start the FastAPI server:

uvicorn main:app --host 0.0.0.0 --port 8000

## Docker

Build the Docker image:

docker build -t zepto-support-assistant .

Run the container:

docker run -p 8000:8000 zepto-support-assistant

## Testing

Policy question:

What is the delivery fee?

The system classifies it as policy and retrieves information from the Zepto policy documents.

General question:

What is 10 + 20?

The system classifies it as general and returns:

10 + 20 = 30
"""

with open("/content/support_assistant/README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("README.md created successfully.")

README.md created successfully.


In [37]:
test_questions = [
    "What is the delivery fee?",
    "Can I cancel my order after it is packed?",
    "What is 10 + 20?"
]

for question in test_questions:
    response = create_support_response(question)

    print("=" * 60)
    print("Question:", question)
    print("Intent:", response.intent)
    print("Answer:", response.answer)
    print("Sources:", response.sources)

Question: What is the delivery fee?
Intent: policy
Answer: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.
Sources: ['Zepto policy documents']
Question: Can I cancel my order after it is packed?
Intent: policy
Answer: Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a p

In [38]:
import os

docs_path = "/content/support_assistant/docs"

files = sorted(os.listdir(docs_path))

print("Number of documents:", len(files))
print("Documents:")

for file in files:
    print(file)

Number of documents: 8
Documents:
doc_01.txt
doc_02.txt
doc_03.txt
doc_04.txt
doc_05.txt
doc_06.txt
doc_07.txt
doc_08.txt


In [39]:
import shutil

shutil.make_archive(
    "/content/support_assistant",
    "zip",
    "/content/support_assistant"
)

print("support_assistant.zip created successfully.")

support_assistant.zip created successfully.
